# Train Chess Figurine Classifier

Train a 5-class CNN to recognize chess piece figurines (K, Q, R, B, N).

**Input:** Manually labeled glyph images in folder structure:
```
glyphs/
├── K/ (King glyphs)
├── Q/ (Queen glyphs)
├── R/ (Rook glyphs)
├── B/ (Bishop glyphs)
└── N/ (Knight glyphs)
```

**Output:** `figurine_classifier.tflite` model (< 500 KB, runs on Flutter app)

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

## Step 2 — Upload labeled glyphs folder

1. Download `glyphs/` folder from the extraction notebook
2. Upload it to Google Drive
3. Set the path below

In [ ]:
import os
from PIL import Image

# ── EDIT THIS: path to labeled glyphs folder ──────────────────────────────
GLYPHS_DIR = '/content/gdrive/MyDrive/entrainement_ocr_echecs/glyphs'

def count_images(folder):
    """Count files that PIL can open as images, regardless of extension."""
    count = 0
    for f in os.listdir(folder):
        path = os.path.join(folder, f)
        if not os.path.isfile(path):
            continue
        try:
            Image.open(path).verify()
            count += 1
        except Exception:
            pass
    return count

if not os.path.exists(GLYPHS_DIR):
    print(f'❌ Glyphs folder not found: {GLYPHS_DIR}')
    print(f'   Upload the glyphs/ folder to Google Drive first')
else:
    print(f'✅ Glyphs folder found')
    print(f'   Contents:')
    for piece in os.listdir(GLYPHS_DIR):
        piece_dir = os.path.join(GLYPHS_DIR, piece)
        if not os.path.isdir(piece_dir):
            continue
        n = count_images(piece_dir)
        all_entries = os.listdir(piece_dir)
        status = '✅' if n > 0 else '❌'
        suffix = f'  (raw entries: {all_entries[:5]})' if n == 0 and all_entries else ''
        print(f'     {status} {piece}: {n} images{suffix}')

## Step 3 — Install dependencies

In [ ]:
!pip install -q tensorflow pillow numpy matplotlib scikit-learn

## Step 4 — Configuration

In [ ]:
import numpy as np
from PIL import Image, ImageOps, ImageFilter
import random
import os
from pathlib import Path
from collections import defaultdict

# Configuration
CLASS_NAMES = ['K', 'Q', 'R', 'B', 'N']  # 5 classes (no pawn)
IMG_SIZE = 32
AUGMENT_PER_IMAGE = 40  # augmentation variants per image
EPOCHS = 60
BATCH_SIZE = 64
VAL_SPLIT = 0.15
TFLITE_PATH = 'figurine_classifier.tflite'

print(f'Classes: {CLASS_NAMES}')
print(f'Input size: {IMG_SIZE}×{IMG_SIZE} grayscale')
print(f'Augmentation: {AUGMENT_PER_IMAGE} variants per image')
print(f'Output: {TFLITE_PATH} (< 500 KB)')

## Step 5 — Load labeled glyph images

In [ ]:
def load_glyphs(glyphs_dir, class_names):
    """Load all labeled glyph images from folder structure."""
    images = []
    labels = []
    counts = defaultdict(int)

    for class_idx, class_name in enumerate(class_names):
        class_dir = os.path.join(glyphs_dir, class_name)
        if not os.path.exists(class_dir):
            print(f'⚠️  {class_name} folder not found')
            continue

        for filename in sorted(os.listdir(class_dir)):
            filepath = os.path.join(class_dir, filename)
            if not os.path.isfile(filepath):
                continue
            try:
                img = Image.open(filepath).convert('L')  # Grayscale
                img = img.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
                arr = np.array(img, dtype=np.float32) / 255.0
                images.append(arr)
                labels.append(class_idx)
                counts[class_name] += 1
            except Exception:
                pass  # skip non-image files silently

    return images, labels, counts

# Load
print(f'Loading glyphs from {GLYPHS_DIR}...\n')
images, labels, counts = load_glyphs(GLYPHS_DIR, CLASS_NAMES)

print(f'✅ Loaded {len(images)} glyph images\n')
print('By class:')
for class_name in CLASS_NAMES:
    count = counts[class_name]
    status = '✅' if count > 0 else '❌'
    print(f'  {status} {class_name}: {count}')

## Step 6 — Data augmentation

In [ ]:
def augment_image(arr, n, size=32):
    """Generate n augmented versions of an image array."""
    results = []
    
    for _ in range(n):
        # Random rotation
        angle = random.uniform(-12, 12)
        img = Image.fromarray((arr * 255).astype(np.uint8))
        img = img.rotate(angle, resample=Image.BICUBIC, fillcolor=255)
        
        # Random scale
        scale = random.uniform(0.80, 1.20)
        new_size = max(4, int(size * scale))
        img = img.resize((new_size, new_size), Image.LANCZOS)
        
        # Center on canvas
        canvas = Image.new('L', (size, size), 255)
        offset = (size - new_size) // 2
        canvas.paste(img, (offset, offset))
        
        # Random flip
        if random.random() > 0.5:
            canvas = ImageOps.mirror(canvas)
        
        # Slight blur
        if random.random() > 0.8:
            canvas = canvas.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.2, 0.6)))
        
        # To array
        aug_arr = np.array(canvas, dtype=np.float32) / 255.0
        
        # Add noise
        aug_arr += np.random.normal(0, 0.03, aug_arr.shape)
        aug_arr = np.clip(aug_arr, 0.0, 1.0)
        
        results.append(aug_arr)
    
    return results

# Augment
print(f'Augmenting {len(images)} images ({AUGMENT_PER_IMAGE} variants each)...\n')

X, y = [], []
for img_arr, label in zip(images, labels):
    augmented = augment_image(img_arr, AUGMENT_PER_IMAGE)
    X.extend(augmented)
    y.extend([label] * len(augmented))

X = np.array(X)[..., np.newaxis]  # Add channel dimension: (N, 32, 32, 1)
y = np.array(y)

print(f'✅ Generated {len(X)} augmented training samples')
print(f'   Shape: {X.shape}')

## Step 7 — Train CNN

In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=VAL_SPLIT, stratify=y, random_state=42
)

print(f'Train: {len(X_train)}  Val: {len(X_val)}')

# Model (5 classes: K, Q, R, B, N)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
    
    tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPool2D(2),
    
    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPool2D(2),
    
    tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.GlobalAveragePooling2D(),
    
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax'),
], name='figurine_classifier')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

# Train
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='val_accuracy'),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-5, monitor='val_accuracy'),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

val_acc = max(history.history['val_accuracy'])
print(f'\n✅ Best validation accuracy: {val_acc:.1%}')

## Step 8 — Export TFLite model

In [ ]:
# Keras model spot check (before TFLite conversion)
print('Keras model spot check:')
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    samples = X_val[y_val == cls_idx]
    if len(samples) > 0:
        sample = samples[0:1].astype(np.float32)
        probs = model.predict(sample, verbose=0)[0]
        pred = CLASS_NAMES[np.argmax(probs)]
        ok = '✅' if pred == cls_name else '⚠'
        print(f'  {ok} {cls_name} → {pred} ({probs.max():.1%})')

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

size_kb = os.path.getsize(TFLITE_PATH) / 1024
print(f'\n✅ Saved: {TFLITE_PATH} ({size_kb:.0f} KB)')

# Sanity check
interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]

print(f'\n   Input : {inp["shape"]}  dtype={inp["dtype"].__name__}')
print(f'   Output: {out["shape"]}  dtype={out["dtype"].__name__}')

# TFLite spot check
print(f'\nTFLite spot check:')
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    samples = X_val[y_val == cls_idx]
    if len(samples) > 0:
        sample = samples[0:1].astype(np.float32)
        interp.set_tensor(inp['index'], sample)
        interp.invoke()
        probs = interp.get_tensor(out['index'])[0]
        pred = CLASS_NAMES[np.argmax(probs)]
        ok = '✅' if pred == cls_name else '⚠'
        print(f'  {ok} {cls_name} → {pred} ({probs.max():.1%})')

## Step 9 — Download model

Download `figurine_classifier.tflite` and copy to your Flutter app:
```
assets/models/figurine_classifier.tflite
```

In [ ]:
from google.colab import files

files.download(TFLITE_PATH)
print(f'✅ Downloaded {TFLITE_PATH}')